# 2026 Revenue Forecast (v2)

**Objective:** Monthly Jan–Dec 2026 revenue forecast with three scenarios (Status Quo, Second-Purchase Push, Lazada Win-back) using gold tables and teammate outputs (DS1, DS3-1, Roopa DS6, Benny BG/NBD).

**Engine:** `scripts/build_forecast_2026.py` (v2 = seasonal acquisition, cohort repeat, BG/NBD subscription curve). Run the full pipeline with `python scripts/run_forecast.py`.

**Outputs:** `outputs/forecast_2026_monthly.csv`, scenario charts, stacked layer chart, acquisition seasonality chart, backtest summary, and `outputs/forecast_assumptions.md`.


## 1. Setup and paths


In [1]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

BASE = Path(".").resolve()
GOLD_DIR = BASE / "medallion" / "gold"
OUTPUT_DIR = BASE / "outputs"
OUTPUT_DIR.mkdir(exist_ok=True)

print("Project root:", BASE)
print("Gold dir exists:", GOLD_DIR.exists())


Project root: D:\Big Uni Files\Customer_Insights\Applied-Data-Group-Project
Gold dir exists: True


## 1b. Refresh teammate metrics (extractors)

Runs the extractor scripts that write forecast-ready JSON into `outputs/`:

- `scripts/extract_ds6_metrics.py` → `outputs/ds6_metrics.json`
- `scripts/extract_ds6_roopa_metrics.py` → `outputs/ds6_roopa_metrics.json`
- `scripts/extract_benny_clv_decay_metrics.py` → `outputs/clv_decay_metrics.json`

Set `RUN_EXTRACTORS = False` below to skip and use whatever JSON is already in `outputs/`.


In [2]:
import subprocess
import sys

RUN_EXTRACTORS = True

EXTRACTOR_SCRIPTS = [
    BASE / "scripts" / "extract_ds6_metrics.py",
    BASE / "scripts" / "extract_ds6_roopa_metrics.py",
    BASE / "scripts" / "extract_benny_clv_decay_metrics.py",
]

if RUN_EXTRACTORS:
    for script in EXTRACTOR_SCRIPTS:
        if not script.exists():
            print(f"[SKIP] Missing: {script.name}")
            continue
        print(f"Running {script.name}...")
        result = subprocess.run(
            [sys.executable, str(script)],
            cwd=str(BASE),
            capture_output=True,
            text=True,
        )
        if result.stdout:
            print(result.stdout.strip())
        if result.returncode != 0:
            if result.stderr:
                print(result.stderr.strip())
            raise RuntimeError(f"Extractor failed: {script.name}")
    print("Extractor outputs refreshed in", OUTPUT_DIR)
else:
    print("Skipping extractors — using existing JSON in", OUTPUT_DIR)


Running extract_ds6_metrics.py...
Wrote D:\Big Uni Files\Customer_Insights\Applied-Data-Group-Project\outputs\ds6_metrics.json
Running extract_ds6_roopa_metrics.py...
Wrote D:\Big Uni Files\Customer_Insights\Applied-Data-Group-Project\outputs\ds6_roopa_metrics.json
Running extract_benny_clv_decay_metrics.py...
Wrote D:\Big Uni Files\Customer_Insights\Applied-Data-Group-Project\outputs\clv_decay_metrics.json
Extractor outputs refreshed in D:\Big Uni Files\Customer_Insights\Applied-Data-Group-Project\outputs


## 2. Constants (DS1, DS3-1, DS6)


In [3]:
# ---------------------------------------------------------------------------
# Constants from teammate notebooks (DS1, DS3-1) + parameters loaded from config
# ---------------------------------------------------------------------------
import json

OVERALL_REPEAT_RATE = 0.2191
SUBSCRIBER_REPEAT_RATE = 0.85
NON_SUBSCRIBER_REPEAT_RATE = 0.195

DEFAULT_FORECAST_PARAMS = {
    "promo_gap_pp": 0.0411,
    "promo_lost_revenue_per_cohort": 2626.0,
    "lazada_winback_conservative": 9776.0,
    "lazada_winback_upside": 43652.79,
    "annual_recovery_pivot": 5253.0,
    "sub_monthly_survival": 0.95,  # proxy until BG/NBD decay is wired in
}


def _load_json_if_exists(path: Path) -> dict:
    if path.exists():
        return json.loads(path.read_text(encoding="utf-8"))
    return {}


def load_forecast_params() -> dict:
    """Load overrides from configs/ + generated outputs/."""
    cfg = _load_json_if_exists(BASE / "configs" / "forecast_2026_params.json")
    ds6 = _load_json_if_exists(OUTPUT_DIR / "ds6_metrics.json")
    roopa = _load_json_if_exists(OUTPUT_DIR / "ds6_roopa_metrics.json")
    clv = _load_json_if_exists(OUTPUT_DIR / "clv_decay_metrics.json")

    clv_override = {}
    if isinstance(clv.get("sub_monthly_survival_proxy"), dict) and "value" in clv["sub_monthly_survival_proxy"]:
        clv_override["sub_monthly_survival"] = clv["sub_monthly_survival_proxy"]["value"]

    roopa_override = {}
    if isinstance(roopa.get("repeat_month_multipliers"), dict):
        roopa_override["repeat_month_multipliers"] = roopa["repeat_month_multipliers"]
    if isinstance(roopa.get("scenario_simulation"), dict):
        sim = roopa["scenario_simulation"]
        if "annual_recovery_pivot" in sim:
            roopa_override["annual_recovery_pivot"] = sim["annual_recovery_pivot"]
    elif "annual_recovery_pivot" in roopa:
        roopa_override["annual_recovery_pivot"] = roopa["annual_recovery_pivot"]

    return {**DEFAULT_FORECAST_PARAMS, **cfg, **ds6, **roopa_override, **clv_override}


FORECAST_PARAMS = load_forecast_params()

CHANNEL_MAP = {
    "DTC": "DTC",
    "Lazada": "Lazada",
    "Shopee": "Shopee",
    "Marketplace": "Other",
    "Other Marketplace": "Other",
    "Draft Order": "Other",
    "Bulk Import": "Other",
    "POS": "Other",
    "Email": "DTC",
    "TikTok": "Other",
    "Shop App": "Other",
    "Affiliate": "Other",
}

FORECAST_CHANNELS = ["DTC", "Lazada", "Shopee", "Other"]

# DS3-1 documented repeat rates (used when computed rates differ slightly)
DOCUMENTED_REPEAT = {
    "DTC": 0.2170,
    "Lazada": 0.2893,
    "Shopee": 0.1832,
    "Other": 0.1700,
}


## 3. Load gold parquets


In [4]:
def load_data():
    cp = pd.read_parquet(GOLD_DIR / "gold_customer_profiles.parquet")
    co = pd.read_parquet(GOLD_DIR / "gold_customer_orders.parquet")
    subs = pd.read_parquet(GOLD_DIR / "gold_subscription_behaviour.parquet")
    cohorts_ch = pd.read_parquet(GOLD_DIR / "gold_retention_cohorts_channel.parquet")
    return cp, co, subs, cohorts_ch


def prepare_customer_base(cp, co):
    cp = cp.copy()
    cp["fc_channel"] = cp["acquisition_channel"].map(CHANNEL_MAP).fillna("Other")
    cp["first_month"] = pd.to_datetime(cp["first_order_date"]).dt.tz_localize(None).dt.to_period("M")
    first_orders = co[co["is_first_order"] == True][["customer_id", "price_total"]].rename(
        columns={"price_total": "first_order_aov"}
    )
    return cp.merge(first_orders, on="customer_id", how="left")


cp, co, subs, cohorts_ch = load_data()
base = prepare_customer_base(cp, co)
print(f"Customers: {len(base):,} | Orders: {len(co):,} | Cohort rows: {len(cohorts_ch):,}")


Customers: 13,885 | Orders: 28,054 | Cohort rows: 695


## 4. Parameter extraction


In [5]:
def extract_parameters(base: pd.DataFrame, subs: pd.DataFrame) -> dict:
    channel_stats = (
        base.groupby("fc_channel")
        .agg(
            repeat_rate_90d=("repeat_purchase_90d", "mean"),
            first_order_aov=("first_order_aov", "mean"),
            customers=("customer_id", "count"),
        )
        .round(4)
    )

    repeat_rates = {}
    first_aov = {}
    for ch in FORECAST_CHANNELS:
        if ch in channel_stats.index:
            repeat_rates[ch] = float(
                DOCUMENTED_REPEAT.get(ch, channel_stats.loc[ch, "repeat_rate_90d"])
            )
            first_aov[ch] = float(channel_stats.loc[ch, "first_order_aov"])
        else:
            repeat_rates[ch] = 0.17
            first_aov[ch] = 60.0

    active_subs = int((~subs["is_churned"].fillna(True)).sum())
    sub_aov = float(subs["avg_order_value"].mean())

    return {
        "overall_repeat_rate": OVERALL_REPEAT_RATE,
        "promo_gap_pp": float(FORECAST_PARAMS["promo_gap_pp"]),
        "promo_lost_revenue_per_cohort": float(FORECAST_PARAMS["promo_lost_revenue_per_cohort"]),
        "subscriber_repeat_rate": SUBSCRIBER_REPEAT_RATE,
        "non_subscriber_repeat_rate": NON_SUBSCRIBER_REPEAT_RATE,
        "lazada_winback_conservative": float(FORECAST_PARAMS["lazada_winback_conservative"]),
        "lazada_winback_upside": float(FORECAST_PARAMS["lazada_winback_upside"]),
        "sub_monthly_survival": float(FORECAST_PARAMS["sub_monthly_survival"]),
        "annual_recovery_pivot": float(FORECAST_PARAMS.get("annual_recovery_pivot", 5253.0)),
        "repeat_rates": repeat_rates,
        "first_aov": first_aov,
        "active_subscribers": active_subs,
        "sub_aov": sub_aov,
        "channel_stats": channel_stats,
    }


In [6]:
params = extract_parameters(base, subs)
params["channel_stats"]


,repeat_rate_90d,first_order_aov,customers
fc_channel,,,
DTC,0.2101,103.0903,5774
Lazada,0.2893,88.0468,3660
Other,0.1679,46.8363,3747
Shopee,0.1832,36.3319,704


## 5. Historical monthly acquisitions (2024–2026)


In [7]:
def historical_monthly_acquisitions(base: pd.DataFrame) -> pd.DataFrame:
    hist = (
        base[base["first_month"] >= "2024-01"]
        .groupby(["first_month", "fc_channel"])
        .size()
        .unstack(fill_value=0)
        .reindex(columns=FORECAST_CHANNELS, fill_value=0)
    )
    return hist

hist = historical_monthly_acquisitions(base)
hist.tail(6)


fc_channel,DTC,Lazada,Shopee,Other
first_month,,,,
2025-10,129,0,95,132
2025-11,131,0,118,257
2025-12,71,0,69,39
2026-01,118,12,82,25
2026-02,185,19,84,33
2026-03,165,11,132,23


## 6. Run forecast engine (v2)

Delegates to `scripts/build_forecast_2026.py` — seasonal acquisition, cohort repeat (Roopa), BG/NBD subscription curve (Benny), and three scenarios. Set `FORECAST_VERSION = "v1"` to compare against the legacy flat run-rate model.


In [8]:
import importlib.util
import sys

FORECAST_VERSION = "v2"  # "v1" for legacy flat run-rate comparison

spec = importlib.util.spec_from_file_location(
    "build_forecast_2026", BASE / "scripts" / "build_forecast_2026.py"
)
bf = importlib.util.module_from_spec(spec)
sys.modules["build_forecast_2026"] = bf
spec.loader.exec_module(bf)

all_scenarios, params, hist, projected = bf.build_forecast(
    version=FORECAST_VERSION,
    cp=cp,
    co=co,
    subs=subs,
    cohorts_ch=cohorts_ch,
    base=base,
    write_outputs=True,
)

status_quo = all_scenarios[all_scenarios["scenario"] == "Status Quo"]
annual_totals = all_scenarios.groupby("scenario")["total_revenue"].sum()
print(f"Forecast version: {FORECAST_VERSION}")
annual_totals


month
2026-01    321.666667
2026-02    321.666667
2026-03    321.666667
2026-04    321.666667
2026-05    321.666667
Freq: M, Name: new_customers, dtype: float64

## 7. Projected acquisition volume (seasonal)

v2 applies a calendar-month seasonal index (2024+) with a mild trend cap and Mar 2026 channel mix anchor.


In [ ]:
projected.groupby("month")["new_customers"].sum()


In [ ]:
status_quo[["month", "new_acq_revenue", "repeat_revenue", "subscription_revenue", "total_revenue"]].head(6)


## 8. Charts

Written to `outputs/` by the forecast engine.


In [ ]:
from IPython.display import Image, display

for name in [
    "forecast_2026_scenarios.png",
    "forecast_2026_delta.png",
    "forecast_2026_stacked_status_quo.png",
    "forecast_2026_acquisition_seasonality.png",
]:
    p = OUTPUT_DIR / name
    if p.exists():
        display(Image(filename=str(p)))


## 9. Q1 2026 backtest (Status Quo vs actuals)


In [ ]:
import sys
sys.path.insert(0, str(BASE / 'scripts'))
from forecast_backtest import run_backtest

backtest_df = run_backtest()
backtest_df


## 10. Assumptions document


In [ ]:
assumptions_path = OUTPUT_DIR / "forecast_assumptions.md"
print(assumptions_path.read_text(encoding="utf-8")[:1200])


## 11. Validation checks


In [ ]:
checks = bf.run_validation(all_scenarios, params, projected, hist, cohorts_ch, FORECAST_VERSION)
for c in checks:
    print(c)


In [ ]:
## 12. Slide-ready summary bullets


totals = all_scenarios.groupby("scenario")["total_revenue"].sum()
sq = totals["Status Quo"]
sp = totals["Second-Purchase Push"]
lz = totals["Lazada Win-back"]
print(f"Status Quo 2026: SGD {sq:,.0f}")
print(f"Second-Purchase Push: SGD {sp:,.0f} (+SGD {sp - sq:,.0f} vs SQ)")
print(f"Lazada Win-back: SGD {lz:,.0f} (+SGD {lz - sq:,.0f} vs SQ)")


In [ ]:
## 13. v2 implementation notes

Phases 0-7 are implemented in `scripts/build_forecast_2026.py`:

- **Phase 0:** `scripts/forecast_backtest.py` - v1 vs v2 Q1 2026 MAPE
- **Phase 1:** Seasonal acquisition projection
- **Phase 2:** Cohort repeat engine (Roopa promo split + holiday multipliers)
- **Phase 3:** BG/NBD subscription curve (Benny extractor)
- **Phase 4:** Three scenarios (Pivot, Lazada mix shift + win-back)
- **Phase 5-7:** Charts, config JSON, `scripts/run_forecast.py` orchestrator


# Config knobs live in configs/forecast_2026_params.json
# Full pipeline: python scripts/run_forecast.py


In [17]:
# (legacy cells consolidated above)


# (legacy cells consolidated above)


In [18]:
# (legacy cells consolidated above)


# (legacy cells consolidated above)


In [19]:
# (legacy cells consolidated above)
